# Querying IDL Metadata and Downloading PDFs from AWS Open Data

This notebook serves as a guided tour of the [IDL] dataset.

This notebook demonstrates a workflow for batch downloading documents from the Industry Documents Library (IDL).

## IDL Data Overview

IDL collaborates with the Truth Initiative on the Truth Tobacco Industry Documents Library and with Johns Hopkins University on the UCSF–JHU Opioid Industry Documents Archive.

The IDL Data on AWS provides access to PDF files, image files, extracted text (OCR) files, and, where available, native files such as Excel spreadsheets, Microsoft Word documents, and PowerPoint presentations. These materials are available for all document records in the collection, enabling researchers and the public to apply computational methods to analyze the documents at scale.

The documents consist primarily of internal corporate records that were made publicly available through ongoing litigation brought by local and state governments and tribal communities against manufacturers, wholesalers, distributors, and pharmacies. The collection includes a wide range of document types and topics, including:

Email communications among sales representatives, physicians, and other healthcare professionals
Internal sales training materials, sales representative data, and compensation strategies
Submissions to regulatory agencies, including consumer guides, brochures, and prescribing information
Graphic designs for product packaging and labeling
Brochures and prescribing publications intended for physicians and the general public
Advertisements and other marketing materials
Correspondence with physicians
Other internal corporate documents

## Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

In the S3 bucket, you will find:

1. An index.html file containing overview information about the dataset.
2. Folders containing files for each document, organized by document ID. For example, files associated with the document record fgdm0000 can be accessed under f/g/d/m/fgdm0000/.
3. Please note that metadata of the document records is **not** hosted on AWS Open Data.  It can be accessed through the [IDL API](https://www.industrydocuments.ucsf.edu/resources/api/) or pre-built [datasets](https://www.industrydocuments.ucsf.edu/resources/global-resources-datasets/). 



## Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?


Our dataset consists of collections of .pdf, .tiff, .zip (native files), .png (thumbnail images), and .ocr (extracted text) files, organized as described in the previous section. 

Please see **IDL Data Overview** section for the kinds of data stored in these files.

Most researchers are interested in the .pdf, .zip, and .ocr files. TIFF files contain the individual images that make up the PDF documents, while PNG files are thumbnail images.


## The process:
1. **Query** `metadata.idl.ucsf.edu` to get document IDs by industry using cursor-based pagination
2. **Build** AWS S3 URLs for each document
3. **Download** PDFs in batches to the local `downloads/` directory
4. **Save** metadata to a CSV file as you go (memory-efficient, batch-by-batch)

**Note:** Documents are processed in batches to keep memory usage constant regardless of dataset size.

## Setup

Install dependencies for querying Solr, downloading files, and data processing.

In [ ]:
# Example dependencies
!pip install requests pandas tqdm matplotlib

import requests
import pandas as pd
from pathlib import Path

## Configure Industry

Choose which industry you want to download documents for.

Available industries: `tobacco`, `opioids`, `chemical`, `drug`, `food`, `fossilfuel`

Edit the `industry` variable below to change your selection.

In [ ]:
# Example: user selects a single industry
# available industries:
#   tobacco
#   opioids
#   chemical
#   drug
#   food
#   fossil fuel
# update industry and/or collection or put in your custom SOLR query 
# to the one you are interested in (using fossil fuel here because it has a small sample size):

industry = "tobacco"
print(f"Selected industry: {industry}")



## Solr Query Function

This function queries a single batch of documents using cursor-based pagination.
Cursor marks ensure stable pagination through large result sets.

In [ ]:
SOLR_ENDPOINT = "https://metadata.idl.ucsf.edu/solr/ltdl3/query"

def query_ids_for_industry(industry, cursor_mark="*", rows=100):
    """Query a single batch of documents for an industry.
    
    Returns a tuple of (docs, next_cursor_mark)
    """
    # These two variables are used to restrict sample size to 200
    # You can adjust them to get different sample size
    # if you could like all document, please change id_high to *
    id_low = 0
    id_high = 199

    params = {
        "q": f"industry:{industry} AND id_int:[{id_low} TO {id_high}]",
        "rows": rows,
        "wt": "json",
        "cursorMark": cursor_mark,
        "sort": "id asc",
    }

    response = requests.get(SOLR_ENDPOINT, params=params)
    print(f"Response status: {response.status_code}")
    print(f"Response headers: {response.headers}")
    print(f"Response text (first 2000 chars): {response.text[:2000]}")
    response.raise_for_status()
    
    data = response.json()
    
    docs = data["response"]["docs"]
    next_cursor_mark = data["nextCursorMark"]
    
    return docs, next_cursor_mark

## Fetch, Build URLs, and Download Documents

Iterates through all documents for the selected industry in batches:
- Queries Solr for a batch of IDs
- Builds S3 URLs for each document
- Downloads PDFs to `downloads/` directory
- Saves metadata to CSV (appends each batch)

This batch-by-batch approach keeps memory usage constant.

In [ ]:
def build_pdf_url(doc_id):
    AWS_BASE_URL = "https://ucsf-idl-dataset.s3.us-east-1.amazonaws.com"
    return f"{AWS_BASE_URL}/{doc_id[0]}/{doc_id[1]}/{doc_id[2]}/{doc_id[3]}/{doc_id}/{doc_id}.pdf"

In [ ]:

def download_pdf(doc_id, pdf_url):
    download_dir = Path("downloads")
    download_dir.mkdir(exist_ok=True)
    output_path = download_dir / f"{doc_id}.pdf"

    response = requests.get(pdf_url, stream=True)

    if response.status_code == 200:
        with open(output_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {output_path}")
    else:
        print(f"Failed: {doc_id} ({response.status_code})")

In [ ]:
# Paginate through all documents and download (batched processing)
cursor_mark = "*"
batch_num = 0
total_processed = 0
csv_file = "idl_metadata_export.csv"

# Remove file if it exists to start fresh
import os
if os.path.exists(csv_file):
    os.remove(csv_file)

while True:
    try:
        docs, next_cursor_mark = query_ids_for_industry(industry, cursor_mark)
    except Exception as e:
        print(f"Error fetching batch: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        break

    if not docs:
        print("No more documents to fetch")
        break

    batch_num += 1
    print(f"\n--- Batch {batch_num} ({len(docs)} documents) ---")

    # Process batch
    batch_data = []
    for doc in docs:
        doc["industry"] = industry
        batch_data.append(doc)
        
        doc_id = doc["id"]
        pdf_url = build_pdf_url(doc_id)
        download_pdf(doc_id, pdf_url)

    # optional: Save batch to CSV (append mode)
    batch_df = pd.DataFrame(batch_data)
    batch_df.to_csv(csv_file, mode='a', header=(batch_num == 1), index=False)
    
    total_processed += len(docs)
    print(f"Saved batch {batch_num} to CSV. Total processed: {total_processed}")

    if next_cursor_mark == cursor_mark:
        print("Reached end of results")
        break

    cursor_mark = next_cursor_mark

print(f"\nCompleted. Total documents processed: {total_processed}")

# Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Please note that we are using a different end point for solr that allows us more flexibility querying the data
SOLR_ENDPOINT_SELECT = "https://metadata.idl.ucsf.edu/solr/ltdl3/select"
# Query with facets on industry_facet
params = {
    "q": "*:*",
    "rows": 0,
    "facet": "true",
    "facet.field": "industry_facet",
    "wt": "json",
}

response = requests.get(SOLR_ENDPOINT_SELECT, params=params)
print(f"Response text (first 1000 chars): {response.text[:1000]}")
response.raise_for_status()
data = response.json()


# Extract facet counts
facet_counts = data["facet_counts"]["facet_fields"]["industry_facet"]

# Convert to dictionary (facet_counts is a flat list alternating code and count)
industry_counts = {}
for i in range(0, len(facet_counts), 2):
    name = facet_counts[i]
    count = facet_counts[i + 1]
    industry_counts[name] = count
    print(f"{name}: {count} documents")

# Create visualization
df = pd.DataFrame(list(industry_counts.items()), columns=["Industry", "Count"])
df = df.sort_values("Count", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(df["Industry"], df["Count"], color="steelblue", edgecolor="navy", alpha=0.7)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel("Industry", fontsize=12, fontweight='bold')
ax.set_ylabel("Number of Documents", fontsize=12, fontweight='bold')
ax.set_title("Document Count by Industry in IDL Dataset", fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(f"\nTotal documents across all industries: {df['Count'].sum():,}")

# What is one question that you have answered using these data? Can you show us how you came to that answer?

One question that has been answered using these data is: How did Juul and related e-cigarette companies use product placement and external partnerships to market their products and increase their credibility?

Researchers answered this question by analyzing internal documents from companies such as Ploom, Pax Labs, and Juul that were available through the Industry Documents Library (IDL). These documents included communications, reports, and other records that provided insight into the companies' marketing strategies. Through this analysis, researchers were able to reconstruct product-placement efforts in music videos, television programs, and films, as well as examine relationships with scientific experts and public-health influencers.

The process involved collecting and organizing large numbers of documents and metadata from multiple sources. As described in the publication, researchers used a combination of manual review and computational methods, including R, to gather data, extract relevant information, and map relationships among organizations, authors, funders, and publishers. By systematically analyzing these connections, they were able to identify patterns and networks that would have been difficult to observe through individual document searches alone.

This example demonstrates how the IDL can be used not only to locate documents but also to conduct large-scale analyses that reveal corporate strategies, professional relationships, and patterns of influence.

# What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

One unanswered research question that could be explored using these data is: How many customer complaints did Juul receive in 2018, and what trends can be identified from those complaints over time? Questions like this cannot easily be answered through the standard IDL website interface, but they become possible when the archive is treated as a dataset for analysis.

Another valuable research question would be to examine relationships among key actors within and across industries. By accessing the underlying data, researchers could generate network graphs that reveal connections between companies, public relations firms, lobbying organizations, and purported grassroots advocacy groups. These analyses could provide insights into how influence and communication networks are structured.

For researchers interested in answering these types of questions, I would recommend working directly with the archived data rather than relying solely on the website interface. Open access to the data allows for bulk downloading and organization of documents in ways that better support research objectives. For example, a researcher could download and analyze all available Juul Slack messages, group conversations by topic or time period, and apply quantitative or qualitative methods to identify patterns that would be difficult to detect through the IDL website alone.
